# Stage 10 - Run the pipeline & export for Power BI (stage by stage)

`src/pipeline.py` is the **headless production version** of everything you built
interactively in `00_build_validate_pipeline`. Running it once rebuilds the whole
warehouse from the raw CSVs and then **exports** every table to files Power BI can read.

This notebook calls the **same functions** as `pipeline.py`, one stage at a time, so
you can watch each step and inspect what it produced. What the script does end-to-end:

| Stage | Function in pipeline.py | What it does | What it writes |
|---|---|---|---|
| 1 | `validate_sources` | Confirm all 7 CSVs exist; fingerprint them | (nothing; returns a list) |
| 2 | `duckdb.connect` | Open the warehouse file | `artifacts/practice_analytics.duckdb` |
| 3 | `ingest_raw` | Load each CSV into `raw.*` | tables in DuckDB |
| 4 | `execute_sql_modules` | Run `01...07.sql` in order (stg -> core -> mart -> qa) | tables in DuckDB |
| 5 | (COPY in `run`) | Export QA tables | `outputs/qa/*.csv` |
| 6 | `export_table` x 25 | Export every Power BI table | `data/processed/power_bi/{csv,parquet}/*` |
| 7 | (metadata in `run`) | Write a run manifest | `artifacts/pipeline_run.json` |

> **Before you run:** shut down the kernel of `00_build_validate_pipeline` first.
> DuckDB allows only **one** read-write connection to the file at a time, so if that
> notebook still holds it open you'll get a "file is being used by another process" error.

## Stage 0 - Configuration and paths

In [1]:
import sys
import time
import json
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

# This notebook lives in practice_analysis/notebooks, so project_root is its parent
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

src_dir = project_root / "src"
sql_dir = project_root / "sql" / "duckdb"
database_path = project_root / "artifacts" / "practice_analytics.duckdb"

# The 7 original CSVs live outside the project (never edited) - see config.example.json
csv_dir = (
    project_root.parent
    / "original_analysis"
    / "5_the_look_ecommerce"
    / "csv_version"
    / "3_thelookecommerce"
)

# Make pipeline.py importable so we reuse its exact functions
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
import pipeline  # noqa: E402

print("project_root :", project_root)
print("csv_dir      :", csv_dir, "(exists:", csv_dir.exists(), ")")
print("sql_dir      :", sql_dir)
print("database     :", database_path)

project_root : c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis
csv_dir      : c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\original_analysis\5_the_look_ecommerce\csv_version\3_thelookecommerce (exists: True )
sql_dir      : c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\sql\duckdb
database     : c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\artifacts\practice_analytics.duckdb


## Stage 1 - Validate the source CSVs

`validate_sources` checks that all 7 required files exist and records each file's size
and a **SHA-256 fingerprint**. The fingerprint is a unique signature of the file's
contents - if the CSV ever changes, the fingerprint changes, so the run manifest can
prove exactly which data version produced the outputs (reproducibility).

In [2]:
inventory = pipeline.validate_sources(csv_dir)
display(pd.DataFrame(inventory))
print("All", len(inventory), "source files present.")

,file,bytes,sha256
0,users.csv,16566424,695dbb4aedc358c28174a515714a9642e1ef792f837fa4...
1,products.csv,4316084,b38f1c006daaebbeb09cd6d0e22068f2a29ea4b48c8a92...
2,orders.csv,9733931,dce6a83f16a270cf69cd3a3b79a3ee25814ecae56d50ad...
3,order_items.csv,18883612,becc82efe86082f29b47dc8a0460b625104a1db3b4acb4...
4,events.csv,384444587,0cb3f3c4a17be80473a47b2ba27fe8052ee0615f935bb2...
5,inventory_events.csv,91956433,8168c7e4cfec08df7e23232af03b1fc94b209d4eae7109...
6,distribution_centers.csv,368,6d9ba37bdb2d56fbeb04a51c624e2e788ea33b6f281d22...


All 7 source files present.


## Stage 2 - Open the DuckDB warehouse

One connection, with two performance settings the script uses (`threads = 4`,
`preserve_insertion_order = false`).

In [3]:
database_path.parent.mkdir(parents=True, exist_ok=True)
connection = duckdb.connect(str(database_path))
connection.execute("SET threads = 4")
connection.execute("SET preserve_insertion_order = false")
run_started = time.perf_counter()
print("Connected to:", database_path)

Connected to: c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\artifacts\practice_analytics.duckdb


## Stage 3 - Ingest the raw layer

`ingest_raw` creates the `raw` schema and loads each CSV into `raw.<table>` with
`read_csv_auto` (auto-detect types, treat empty string as NULL). This is the only step
that touches the CSV files; everything after works purely inside DuckDB.

In [4]:
pipeline.ingest_raw(connection, csv_dir)

raw_counts = {
    t: connection.execute(f"SELECT COUNT(*) FROM raw.{t}").fetchone()[0]
    for t in pipeline.SOURCE_TABLES
}
display(pd.DataFrame(
    [{"table": f"raw.{t}", "rows": n} for t, n in raw_counts.items()]
))

Loading raw.users
Loading raw.products
Loading raw.orders
Loading raw.order_items
Loading raw.events


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loading raw.inventory_events
Loading raw.distribution_centers


,table,rows
0,raw.users,100000
1,raw.products,29120
2,raw.orders,124814
3,raw.order_items,180862
4,raw.events,2420661
5,raw.inventory_events,488146
6,raw.distribution_centers,10


## Stage 4 - Build staging, core, marts, and QA

`execute_sql_modules` runs every `.sql` file in `sql/duckdb` **in filename order**
(`01_staging` -> `02_core_dimensions` -> ... -> `07_quality_and_snapshots`). This is the
same SQL you wrote and tested in notebook 00 - here it runs top to bottom in one shot.
The returned table shows how long each module took.

In [5]:
module_runs = pipeline.execute_sql_modules(connection, sql_dir)
display(pd.DataFrame(module_runs))

Executing 01_staging.sql
Executing 02_core_dimensions.sql
Executing 03_core_facts.sql
Executing 04_mart_acquisition_funnel.sql
Executing 05_mart_commercial_customer.sql
Executing 06_mart_operations_inventory.sql
Executing 07_quality_and_snapshots.sql


,module,seconds
0,01_staging.sql,0.038
1,02_core_dimensions.sql,0.996
2,03_core_facts.sql,2.233
3,04_mart_acquisition_funnel.sql,0.120
4,05_mart_commercial_customer.sql,1.186
5,06_mart_operations_inventory.sql,0.087
6,07_quality_and_snapshots.sql,2.575


## Stage 5 - Export the QA tables

The two QA audit tables are written to `outputs/qa/` as CSV so they can be reviewed or
version-controlled outside the database.

In [6]:
qa_dir = project_root / "outputs" / "qa"
qa_dir.mkdir(parents=True, exist_ok=True)

for table in ["test_results", "metric_reconciliation"]:
    target = qa_dir / f"{table}.csv"
    connection.execute(
        f"COPY qa.{table} TO '{pipeline.sql_literal(target)}' "
        "(HEADER, DELIMITER ',')"
    )
    print("Wrote", target)

Wrote c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\outputs\qa\test_results.csv
Wrote c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\outputs\qa\metric_reconciliation.csv


## Stage 6 - Export the 25 Power BI tables (CSV + Parquet)

`export_table` writes each core/mart table twice, into
`data/processed/power_bi/csv/` and `data/processed/power_bi/parquet/`.

- **CSV** - universal, human-readable, easy to eyeball.
- **Parquet** - columnar + compressed (ZSTD); much smaller and faster for Power BI to load.

`POWER_BI_TABLES` is the curated list of exactly which tables ship to the dashboard
(10 core + 15 marts). The `raw` and `stg` layers are intentionally *not* exported -
they're internal build steps, not analysis outputs.

In [7]:
exports = [
    pipeline.export_table(connection, project_root, schema, table)
    for schema, table in pipeline.POWER_BI_TABLES
]
exports_df = pd.DataFrame(exports)
display(exports_df)
print(f"Exported {len(exports)} tables, {exports_df['rows'].sum():,} total rows.")

,schema,table,rows
0,core,dim_date,1994
1,core,dim_customer,100000
2,core,dim_product,29120
3,core,dim_distribution_center,10
4,core,dim_session_traffic_source,5
5,core,dim_acquisition_source,5
6,core,fact_order,124814
7,core,fact_order_item,180862
8,core,fact_session,680862
9,core,fact_inventory,488146


Exported 25 tables, 1,691,576 total rows.


## Stage 6b - Export the analysis/ folder (readable mart CSVs)

Mirrors `new_analysis`: a curated set of **already-aggregated, readable** tables written
as plain CSV to `data/processed/analysis/`, for opening in Excel / pandas without building
a data model. It deliberately **excludes** `customer_360` (1 row per customer, too granular)
and `funnel_stage_channel` (a Power BI visual-support table), and includes the KPI scorecard
`qa.metric_snapshot` (practice's equivalent of new_analysis's `mart.executive_snapshot`).

In [8]:
# Curated, readable summary CSVs -> data/processed/analysis/
analysis_root = project_root / "data" / "processed" / "analysis"
analysis_root.mkdir(parents=True, exist_ok=True)

analysis_exports = []
for schema, table in pipeline.ANALYSIS_TABLES:
    csv_path = analysis_root / f"{table}.csv"
    connection.execute(
        f"COPY (SELECT * FROM {schema}.{table}) "
        f"TO '{pipeline.sql_literal(csv_path)}' (HEADER, DELIMITER ',')"
    )
    rows = connection.execute(
        f"SELECT COUNT(*) FROM {schema}.{table}"
    ).fetchone()[0]
    analysis_exports.append({"schema": schema, "table": table, "rows": int(rows)})

display(pd.DataFrame(analysis_exports))
print(f"Exported {len(analysis_exports)} analysis tables to {analysis_root}")

,schema,table,rows
0,qa,metric_snapshot,1
1,mart,channel_quality,5
2,mart,funnel_monthly_channel,325
3,mart,acquisition_source_value,5
4,mart,cart_abandonment_segments,22
5,mart,sales_monthly,65
6,mart,product_performance_category,36
7,mart,product_performance_brand,2755
8,mart,geography_performance_country,15
9,mart,cohort_retention,1936


Exported 14 analysis tables to c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\data\processed\analysis


## Stage 7 - Write the run manifest and close

The manifest (`artifacts/pipeline_run.json`) records what this run did: source
fingerprints, raw row counts, per-module timings, and export row counts. It's the
paper trail that makes the run auditable and reproducible.

In [9]:
metadata = {
    "database_path": str(database_path),
    "source_csv_dir": str(csv_dir.resolve()),
    "source_inventory": inventory,
    "raw_row_counts": {k: int(v) for k, v in raw_counts.items()},
    "sql_modules": module_runs,
    "exports": exports,
    "analysis_exports": analysis_exports,
    "elapsed_seconds": round(time.perf_counter() - run_started, 3),
    "session_source": "events.csv grouped by session_id",
    "legacy_dim_sessions_required": False,
}

metadata_path = project_root / "artifacts" / "pipeline_run.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

connection.close()
print("Wrote", metadata_path)
print("Elapsed:", metadata["elapsed_seconds"], "seconds")
print("Connection closed.")

Wrote c:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\artifacts\pipeline_run.json
Elapsed: 21.972 seconds
Connection closed.


## Stage 8 - Verify what landed on disk

Confirm the `data/` and `outputs/` folders now look like the reference `new_analysis`.

In [10]:
def show_tree(base: Path):
    base = base.resolve()
    if not base.exists():
        print(base, "(missing)")
        return
    rows = []
    for p in sorted(base.rglob("*")):
        if p.is_file():
            rows.append({
                "file": str(p.relative_to(project_root)),
                "KB": round(p.stat().st_size / 1024, 1),
            })
    display(pd.DataFrame(rows))

print("=== outputs/ ===")
show_tree(project_root / "outputs")
print("=== data/processed/ ===")
show_tree(project_root / "data" / "processed")

=== outputs/ ===


,file,KB
0,outputs\qa\metric_reconciliation.csv,1.0
1,outputs\qa\test_results.csv,2.6


=== data/processed/ ===


,file,KB
0,data\processed\analysis\acquisition_source_val...,0.6
1,data\processed\analysis\cart_abandonment_segme...,1.4
2,data\processed\analysis\channel_quality.csv,0.7
3,data\processed\analysis\cohort_retention.csv,76.0
4,data\processed\analysis\delivery_performance_d...,2.3
...,...,...
59,data\processed\power_bi\parquet\operations_mon...,7.7
60,data\processed\power_bi\parquet\product_perfor...,89.0
61,data\processed\power_bi\parquet\product_perfor...,4.6
62,data\processed\power_bi\parquet\return_risk_se...,2.8
